In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[1]   # move up from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))
from proteins.data.datasets import ESMCSingleDS, SingleSequenceDS
from os.path import join, basename
import biotite.database.rcsb as rcsb
import biotite.structure.io.pdbx as pdbx
import biotite.structure as struc
import numpy as np

In [ ]:
data_name = 'IEDB_Jespersen'
model_name = 'esmc_300m'
base_data_dir = Path.cwd().parents[0] / 'data' / 'data_files'
dataset = SingleSequenceDS(data_name, save_dir=base_data_dir)

In [ ]:
from biotite.database.rcsb import SequenceQuery, search, count
from biotite.sequence import ProteinSequence, align
from biotite.application.muscle import Muscle5App

sequences = dataset.unique_sequences

group = dataset.data[dataset.data['cluster']==3]
group_seqs = [ProteinSequence(seq) for seq in group['Sequence']]
matrix = align.SubstitutionMatrix.std_protein_matrix()
alignments = align.align_optimal(group_seqs[0], group_seqs[1], matrix, local=False)

app = Muscle5App(group_seqs)
app.start()
app.join()
msa_alignment = app.get_alignment()
print(msa_alignment)

seq = 'MKVTGIFLLSALALLSLSGNTGADSLGREAKCYNELNGCTKIYDPVCGTDGNTYPNECVLCFENRKRQTSILIQKSGPC'

# Create the query
query = SequenceQuery(
    sequence        = seq,
    scope           = "protein",
    min_identity    = 0.99,
    max_expect_value = 1e-5
)

# Run the search
pdb_ids = search(query)

print(f"Found {len(pdb_ids)} similar structures")
print("First few hits:", pdb_ids[:10] if pdb_ids else "No hits")

In [ ]:

file_paths = rcsb.fetch(pdb_ids, "bcif", dataset.base_dir / 'pdb_files')

print("Downloaded files:")
for path in file_paths:
    print("  ", basename(path))

In [ ]:
for pdb_id, path in zip(pdb_ids, file_paths):
    print(f"\n=== {pdb_id} ===")

    bcif = pdbx.BinaryCIFFile.read(path)
    array = pdbx.get_structure(bcif, model=1)

    # Sequence from metadata
    seqs = pdbx.get_sequence(bcif)
    for ch, seq in seqs.items():
        print(f"  Chain {ch}: {len(seq)} aa")

    if "C" in np.unique(array.chain_id):
        chain = array[array.chain_id == "C"]
        chain = chain[~chain.hetero & ~chain.res_name.isin(["HOH", "DMS", "EDO"])]  # clean

        # Sequence from atoms
        seq_atoms = struc.to_sequence(chain)
        print(f"  Chain A atoms: {len(seq_atoms)} residues")

        # SASA
        sasa_atoms = struc.sasa(chain, probe_radius=1.4, vdw_radii="ProtOr")
        sasa_res = struc.apply_residue_wise(chain, sasa_atoms, np.sum)

        print(f"  Avg residue SASA: {np.mean(sasa_res):.1f} Å²")
        print(f"  Max residue SASA : {np.max(sasa_res):.1f} Å² (res {np.argmax(sasa_res)+1})")